# EDA, preprocessing and train/valid/test split

В этом ноутбуке выполняется первичный анализ датасета `diabetes.csv`, проверка структуры данных, анализ целевой переменной, базовая предобработка и разбиение данных на train, validation и test выборки.

In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

In [11]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "diabetes.csv"

df = pd.read_csv(DATA_PATH)

## Первичный анализ структуры датасета

In [12]:
print("Dataset path:")
print(DATA_PATH)

print("\nDataset shape:")
print(df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nColumns:")
display(pd.DataFrame({"column": df.columns.tolist()}))

print("\nData types:")
display(df.dtypes.to_frame(name="dtype"))

print("\nInfo:")
df.info()

print("\nMissing values:")
display(df.isna().sum().to_frame(name="missing_count"))

print("\nDescriptive statistics:")
display(df.describe())

print("\nTarget distribution:")
display(df["Outcome"].value_counts().to_frame(name="count"))

print("\nTarget distribution, percent:")
display((df["Outcome"].value_counts(normalize=True) * 100).round(2).to_frame(name="percent"))

Dataset path:
c:\Users\Qekqq\Desktop\Devops\devops_hw_1\data\raw\diabetes.csv

Dataset shape:
(768, 9)

First 5 rows:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1



Columns:


,column
0,Pregnancies
1,Glucose
2,BloodPressure
3,SkinThickness
4,Insulin
5,BMI
6,DiabetesPedigreeFunction
7,Age
8,Outcome



Data types:


...

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000



Target distribution:


,count
Outcome,
0,500
1,268



Target distribution, percent:


,percent
Outcome,
0,65.1
1,34.9


## Проверка нулевых значений

В датасете отсутствуют явные пропуски `NaN`, однако для некоторых медицинских признаков значение `0` может означать отсутствующее или некорректное измерение. Поэтому дополнительно проверим количество нулевых значений в каждом признаке.

In [13]:
zero_counts = (df == 0).sum()
zero_percent = ((df == 0).mean() * 100).round(2)

zero_summary = pd.DataFrame({
    "zero_count": zero_counts,
    "zero_percent": zero_percent
})

display(zero_summary)

,zero_count,zero_percent
Pregnancies,111,14.45
Glucose,5,0.65
BloodPressure,35,4.56
SkinThickness,227,29.56
Insulin,374,48.70
BMI,11,1.43
DiabetesPedigreeFunction,0,0.00
Age,0,0.00
Outcome,500,65.10


## Вывод по нулевым значениям

В датасете отсутствуют явные пропуски `NaN`, однако для части медицинских признаков значение `0` является некорректным с точки зрения предметной области.

Значение `0` допустимо для признака `Pregnancies`, так как количество беременностей может быть равно нулю. Также значение `0` допустимо для целевой переменной `Outcome`, поскольку оно обозначает отрицательный класс.

Для признаков `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` и `BMI` значение `0` будем рассматривать как скрытый пропуск. Удалять строки с такими значениями нецелесообразно, так как в признаках `SkinThickness` и `Insulin` доля нулевых значений достаточно высокая. Поэтому на этапе предобработки такие нули будут заменены на пропуски, а затем заполнены медианными значениями, рассчитанными на обучающей выборке.

In [14]:
zero_as_missing_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

zero_as_missing_columns

['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

In [15]:
TARGET_COLUMN = "Outcome"

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
display(y.head())

Features shape: (768, 8)
Target shape: (768,)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33


0    1
1    0
2    1
3    0
4    1
Name: Outcome, dtype: int64

In [16]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 57

# Сначала отделяем test: 15%
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=y
)

# Теперь из оставшихся 85% отделяем validation.
# Нужно получить valid = 15% от всего датасета.
# 0.15 / 0.85 ≈ 0.1765
valid_size_from_train_valid = 0.15 / 0.85

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid,
    y_train_valid,
    test_size=valid_size_from_train_valid,
    random_state=RANDOM_STATE,
    stratify=y_train_valid
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)
print("y_test:", y_test.shape)

X_train: (536, 8)
X_valid: (116, 8)
X_test: (116, 8)
y_train: (536,)
y_valid: (116,)
y_test: (116,)


In [17]:
split_distribution = pd.DataFrame({
    "train_count": y_train.value_counts(),
    "valid_count": y_valid.value_counts(),
    "test_count": y_test.value_counts(),
    "train_percent": (y_train.value_counts(normalize=True) * 100).round(2),
    "valid_percent": (y_valid.value_counts(normalize=True) * 100).round(2),
    "test_percent": (y_test.value_counts(normalize=True) * 100).round(2),
})

display(split_distribution)

,train_count,valid_count,test_count,train_percent,valid_percent,test_percent
Outcome,,,,,,
0,349,75,76,65.11,64.66,65.52
1,187,41,40,34.89,35.34,34.48


In [ ]:
# Замена нулей на NaN и заполнение медианами
import numpy as np

X_train_processed = X_train.copy()
X_valid_processed = X_valid.copy()
X_test_processed = X_test.copy()

# Заменяем медицински некорректные нули на NaN
for dataset in [X_train_processed, X_valid_processed, X_test_processed]:
    dataset[zero_as_missing_columns] = dataset[zero_as_missing_columns].replace(0, np.nan)

# Медианы считаем только на train
train_medians = X_train_processed[zero_as_missing_columns].median()

print("Train medians for imputation:")
display(train_medians.to_frame(name="median_value"))

# Заполняем пропуски медианами train
for dataset in [X_train_processed, X_valid_processed, X_test_processed]:
    dataset[zero_as_missing_columns] = dataset[zero_as_missing_columns].fillna(train_medians)

Train medians for imputation:


,median_value
Glucose,117.0
BloodPressure,72.0
SkinThickness,30.0
Insulin,126.0
BMI,32.4


In [ ]:
# Проверка после обработки
preprocessing_check = pd.DataFrame({
    "train_missing_after": X_train_processed[zero_as_missing_columns].isna().sum(),
    "valid_missing_after": X_valid_processed[zero_as_missing_columns].isna().sum(),
    "test_missing_after": X_test_processed[zero_as_missing_columns].isna().sum(),
    "train_zero_after": (X_train_processed[zero_as_missing_columns] == 0).sum(),
    "valid_zero_after": (X_valid_processed[zero_as_missing_columns] == 0).sum(),
    "test_zero_after": (X_test_processed[zero_as_missing_columns] == 0).sum(),
})

display(preprocessing_check)

,train_missing_after,valid_missing_after,test_missing_after,train_zero_after,valid_zero_after,test_zero_after
Glucose,0,0,0,0,0,0
BloodPressure,0,0,0,0,0,0
SkinThickness,0,0,0,0,0,0
Insulin,0,0,0,0,0,0
BMI,0,0,0,0,0,0


In [20]:
# Собираем обратно признаки + target

train_data = X_train_processed.copy()
train_data[TARGET_COLUMN] = y_train

valid_data = X_valid_processed.copy()
valid_data[TARGET_COLUMN] = y_valid

test_data = X_test_processed.copy()
test_data[TARGET_COLUMN] = y_test

print("Train data:", train_data.shape)
print("Valid data:", valid_data.shape)
print("Test data:", test_data.shape)

display(train_data.head())

Train data: (536, 9)
Valid data: (116, 9)
Test data: (116, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
420,1,119.0,88.0,41.0,170.0,45.3,0.507,26,0
531,0,107.0,76.0,30.0,126.0,45.3,0.686,24,0
340,1,130.0,70.0,13.0,105.0,25.9,0.472,22,0
608,0,152.0,82.0,39.0,272.0,41.5,0.270,27,0
177,0,129.0,110.0,46.0,130.0,67.1,0.319,26,1


In [21]:
# Сохраняем обработанные данные

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

train_data.to_csv(PROCESSED_DATA_DIR / "train.csv", index=False)
valid_data.to_csv(PROCESSED_DATA_DIR / "valid.csv", index=False)
test_data.to_csv(PROCESSED_DATA_DIR / "test.csv", index=False)

print("Processed datasets saved:")
print(PROCESSED_DATA_DIR / "train.csv")
print(PROCESSED_DATA_DIR / "valid.csv")
print(PROCESSED_DATA_DIR / "test.csv")

Processed datasets saved:
c:\Users\Qekqq\Desktop\Devops\devops_hw_1\data\processed\train.csv
c:\Users\Qekqq\Desktop\Devops\devops_hw_1\data\processed\valid.csv
c:\Users\Qekqq\Desktop\Devops\devops_hw_1\data\processed\test.csv


## Итог по первичной обработке данных

В ходе первичного анализа было установлено, что датасет содержит 768 наблюдений и 9 колонок. Целевая переменная — `Outcome`, остальные 8 колонок используются как признаки модели.

Явные пропуски `NaN` в данных отсутствуют. Однако для признаков `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` и `BMI` значение `0` является некорректным с точки зрения предметной области и было рассмотрено как скрытый пропуск.

Данные были разделены на обучающую, валидационную и тестовую выборки в пропорции 70% / 15% / 15%. Так как классы целевой переменной несбалансированы, при разбиении использовалась стратификация по `Outcome`.

Для предотвращения утечки данных медианные значения для заполнения скрытых пропусков были рассчитаны только на обучающей выборке. Затем эти медианы были применены к обучающей, валидационной и тестовой выборкам.

Обработанные данные сохранены в папку `data/processed/`:

- `train.csv`
- `valid.csv`
- `test.csv`